# **Redes Neuronales 2**
## Práctica 8: Transformers y mecanismos de atención

**Temas:**
- Tokenización y modelos de lenguaje enmascarados (BERT).
- Fine-tuning: continuar el preentrenamiento sobre un corpus propio.
- Fine-tuning para clasificación de texto.
- Transformers en visión (ViT) y visualización de la atención.

Usamos modelos preentrenados de [Hugging Face](https://huggingface.co/). Para más notebooks de aplicaciones: https://huggingface.co/transformers/notebooks.html

In [ ]:
# En Colab hace falta instalar dependencias (local: ya instaladas)
import importlib, subprocess, sys
for pkg, mod in [('transformers', 'transformers'), ('datasets', 'datasets')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

d:\facuult\REDES_NEURONALES_II\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("dispositivo:", device)

dispositivo: cuda


## Parte 1: modelos de lenguaje enmascarados (provisto, ilustrativo)

Cargamos un **BERT** preentrenado y su tokenizador. BERT fue preentrenado prediciendo tokens **enmascarados** — la misma idea de "tapar y predecir" que vimos con los auto-codificadores por eliminación de ruido (Práctica 5).

In [5]:
from transformers import BertTokenizer, BertForMaskedLM

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased").to(device)

W1026 08:58:43.875000 18304 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 238.00it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
print(tokenizer.tokenize("the sailors were lured"))   # ['the', 'sailors', 'were', 'lu', '##red']
print(tokenizer.encode("the sailors were lured"))     # [101, 1996, ..., 102]

['the', 'sailors', 'were', 'lured']
[101, 1996, 11279, 2020, 26673, 102]


### Tokenización

Del tokenizador recibimos `input_ids` (la secuencia de tokens), `token_type_ids` y `attention_mask`. Los tokens especiales importan: `[CLS]`=101 abre la secuencia, `[SEP]`=102 la cierra, `[MASK]`=103 es la máscara y 0 es padding.

In [7]:
texto = ("After Abraham Lincoln won the November 1860 presidential "
         "election on an anti-slavery platform, an initial seven "
         "slave states declared their secession from the country "
         "to form the Confederacy.")

inputs = tokenizer(texto, return_tensors="pt")
print(inputs.keys())
print(tokenizer.convert_ids_to_tokens(inputs.input_ids[0])[:12])

KeysView({'input_ids': tensor([[  101,  2044,  8181,  5367,  2180,  1996,  2281,  7313,  4883,  2602,
          2006,  2019,  3424,  1011,  8864,  4132,  1010,  2019,  3988,  2698,
          6658,  2163,  4161,  2037, 22965,  2013,  1996,  2406,  2000,  2433,
          1996, 18179,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})
['[CLS]', 'after', 'abraham', 'lincoln', 'won', 'the', 'november', '1860', 'presidential', 'election', 'on', 'an']


---
## Ejercicio 1: enmascaramiento estilo BERT

Implementar la estrategia de enmascaramiento: seleccionar al azar el **15%** de los tokens y reemplazarlos por `[MASK]` (103), **sin tocar** los tokens especiales (`[CLS]`=101, `[SEP]`=102, padding=0). Las `labels` deben ser los `input_ids` originales (así la pérdida compara la predicción con el token verdadero).

Luego pasar la entrada enmascarada por el modelo y reportar la pérdida. Decodificar qué predijo el modelo en las posiciones enmascaradas.

In [8]:
inputs["labels"] = inputs.input_ids.detach().clone()

# máscara aleatoria del 15%, evitando tokens especiales
rand = torch.rand(inputs.input_ids.shape)
mask_arr = (rand < 0.15) * (inputs.input_ids != 101) \
                        * (inputs.input_ids != 102) * (inputs.input_ids != 0)
seleccion = torch.flatten(mask_arr[0].nonzero()).tolist()
inputs.input_ids[0, seleccion] = 103          # token [MASK]

inputs = {k: v.to(device) for k, v in inputs.items()}

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
print("pérdida (sin fine-tuning):", outputs.loss.item())

# ¿qué predijo en las posiciones enmascaradas?
pred_ids = outputs.logits[0].argmax(-1)
originales = inputs["labels"][0, seleccion]
predichos = pred_ids[seleccion]
for o, p in zip(originales, predichos):
    print(f"original: {tokenizer.decode([o]):15s} -> predicho: {tokenizer.decode([p])}")

pérdida (sin fine-tuning): 1.1808420419692993
original: election        -> predicho: election
original: secession       -> predicho: secession
original: form            -> predicho: join
original: confederacy     -> predicho: confederacy


In [12]:
print(inputs["labels"])

tensor([[  101,  2044,  8181,  5367,  2180,  1996,  2281,  7313,  4883,  2602,
          2006,  2019,  3424,  1011,  8864,  4132,  1010,  2019,  3988,  2698,
          6658,  2163,  4161,  2037, 22965,  2013,  1996,  2406,  2000,  2433,
          1996, 18179,  1012,   102]], device='cuda:0')


---
## Parte 2: fine-tuning del modelo de lenguaje (provisto)

Continuamos el preentrenamiento enmascarado sobre otro corpus, La Comunidad del Anillo: https://drive.google.com/uc?id=1SbnXl4Cn3eKHiWQysi35P9cGST02CHJX


In [ ]:
!gdown 1SbnXl4Cn3eKHiWQysi35P9cGST02CHJX

Downloading...
From: https://drive.google.com/uc?id=1SbnXl4Cn3eKHiWQysi35P9cGST02CHJX
To: /content/felowship_of_the_ring.txt
100% 1.00M/1.00M [00:00<00:00, 10.8MB/s]


In [ ]:
with open('felowship_of_the_ring.txt', 'r') as fp:
    text_ft = fp.read().split('\n')

len(text_ft)

7744

In [ ]:
text_ft = text_ft[:1000] # Si demora mucho nos quedamos con una parte del libro para que no sea tan pesado
len(text_ft)

1000

In [ ]:
inputs_ft = tokenizer(text_ft, return_tensors='pt', max_length=512, truncation=True, padding='max_length')

inputs_ft.input_ids.size()

inputs_ft["labels"] = inputs_ft.input_ids.detach().clone()

In [ ]:
from transformers import TrainingArguments, Trainer

# enmascaramos el 15% (igual que el Ejercicio 1, ahora por filas)
rand = torch.rand(inputs_ft.input_ids.shape)
mask_arr = (rand < 0.15) * (inputs_ft.input_ids != 101) \
                          * (inputs_ft.input_ids != 102) * (inputs_ft.input_ids != 0)
for i in range(inputs_ft.input_ids.shape[0]):
    sel = torch.flatten(mask_arr[i].nonzero()).tolist()
    inputs_ft.input_ids[i, sel] = 103

class CorpusDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}
    def __len__(self):
        return len(self.encodings.input_ids)

dataset_ft = CorpusDataset(inputs_ft)

args = TrainingArguments(output_dir="out_mlm", per_device_train_batch_size=8,
                         num_train_epochs=1, logging_steps=10, report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=dataset_ft)
trainer.train()

Step,Training Loss
10,0.004336
20,0.007489
30,0.009594
40,0.007708
50,0.008696
60,0.008882
70,0.010857
80,0.010596
90,0.018446
100,0.015163


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=0.012166653096675873, metrics={'train_runtime': 143.9466, 'train_samples_per_second': 6.947, 'train_steps_per_second': 0.868, 'total_flos': 263204812800000.0, 'train_loss': 0.012166653096675873, 'epoch': 1.0})

In [ ]:
# Ejemplos de predicción sobre palabras enmascaradas, ahora con el modelo ajustado al libro.
# (`inputs_ft.input_ids` ya viene enmascarado de la celda anterior; `labels` guarda el original.)
CLS, SEP, MASK, PAD = 101, 102, 103, 0
N_ORACIONES = 10

largos = inputs_ft.attention_mask.sum(1)                  # tokens reales por fila
n_masc = (inputs_ft.input_ids == MASK).sum(1)             # máscaras por fila
# elegimos oraciones ni muy cortas ni muy largas, y con varias palabras tapadas
candidatas = [i for i in range(len(largos))
              if 20 <= largos[i] <= 60 and n_masc[i] >= 3][:N_ORACIONES]

lote = {k: v[candidatas].to(device) for k, v in inputs_ft.items()}
# la pérdida solo tiene sentido en las posiciones tapadas: en el resto (incluido el
# padding) la etiqueta es una copia de la entrada y se predice sin esfuerzo
lote["labels"] = lote["labels"].masked_fill(lote["input_ids"] != MASK, -100)

model.eval()
with torch.no_grad():
    salida_ft = model(**lote)
print(f"pérdida en los tokens enmascarados: {salida_ft.loss.item():.4f}\n")

pred_ids_ft = salida_ft.logits.argmax(-1).cpu()           # (N_ORACIONES, 512)


def unir_piezas(piezas):
    """Reconstruye el texto uniendo los word-pieces de BERT (los '##')."""
    texto = ""
    for p in piezas:
        if p.startswith("##"):
            texto += p[2:]
        else:
            texto += (" " if texto else "") + p
    return texto


aciertos, total = 0, 0
for fila, idx in enumerate(candidatas):
    orig = inputs_ft["labels"][idx]                       # ids originales (sin enmascarar)
    ent = inputs_ft.input_ids[idx]                        # ids con [MASK]
    pred = pred_ids_ft[fila]
    largo = int(largos[idx])

    con_huecos, con_respuestas = [], []
    for j in range(1, largo - 1):                         # sin [CLS] ni [SEP]
        tok = tokenizer.convert_ids_to_tokens(int(orig[j]))
        if int(ent[j]) == MASK:
            pref = "##" if tok.startswith("##") else ""   # para no partir la palabra
            real = tok[2:] if pref else tok
            adiv = tokenizer.convert_ids_to_tokens(int(pred[j])).removeprefix("##")
            con_huecos.append(pref + "____")
            con_respuestas.append(f"{pref}{real} ({adiv})")
            aciertos += int(orig[j] == pred[j]); total += 1
        else:
            con_huecos.append(tok)
            con_respuestas.append(tok)

    print(f"--- oración {fila + 1} (línea {idx} del libro) ---")
    print("  enmascarada:", unir_piezas(con_huecos))
    print("  real (pred):", unir_piezas(con_respuestas))
    print()

print(f"tokens enmascarados acertados: {aciertos}/{total} ({aciertos / total:.1%})")

pérdida en los tokens enmascarados: 0.4151

--- oración 1 (línea 31 del libro) ---
  enmascarada: many , however , may ____ to ____ more about this remarkable ____ from the ____ ____ while some may not possess the earlier book . for such readers a few notes on the more important points are here collected from hobbit - lore ____ and the ____ adventure is briefly recalled .
  real (pred): many , however , may wish (wish) to know (know) more about this remarkable people (people) from the outset (outset) , (,) while some may not possess the earlier book . for such readers a few notes on the more important points are here collected from hobbit - lore , (,) and the first (first) adventure is briefly recalled .

--- oración 2 (línea 158 del libro) ---
  enmascarada: ‘ it ____ have to be paid ____ , ’ ____ said . ‘ ____ isn ’ t natural , and trouble will come of it ! ’
  real (pred): ‘ it will (will) have to be paid for (for) , ’ they (he) said . ‘ it (it) isn ’ t natural , and trouble will co

---
## Ejercicio 2: fine-tuning para clasificación

Ahora usamos un BERT con **cabeza de clasificación** para predecir la cantidad de estrellas (1–5) de reseñas de **Yelp**. Se provee la carga de datos y la tokenización.

- a) Completar el bucle de entrenamiento (1 época) y una función de evaluación que calcule la accuracy sobre el conjunto de validación.
- b) Repetir el fine-tuning entrenando **solo la cabeza** (congelar `bert.*`): ¿cuántos parámetros entrenables quedan? ¿Cómo cambia la accuracy?
- c) ¿Qué estrategias de _learning rate_ conviene usar? (warmup + decaimiento lineal es lo estándar: `transformers.get_scheduler`).

In [ ]:
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

# torchvision >= 0.23 eliminó VideoReader, pero el formateador "torch" de datasets
# todavía lo importa. Como acá no hay videos, desactivamos esa rama.
datasets.config.TORCHVISION_AVAILABLE = False

tokenizer_cls = AutoTokenizer.from_pretrained("bert-base-cased")

train_ds = load_dataset("Yelp/yelp_review_full", split="train[:2000]")
val_ds = load_dataset("Yelp/yelp_review_full", split="test[:500]")

def tokenizar(ejemplos):
    return tokenizer_cls(ejemplos["text"], padding="max_length",
                         truncation=True, max_length=128)

def preparar(ds):
    ds = ds.map(tokenizar, batched=True)
    ds = ds.remove_columns(["text"]).rename_column("label", "labels")
    ds.set_format("torch")
    return ds

train_loader_cls = DataLoader(preparar(train_ds), shuffle=True, batch_size=16)
val_loader_cls = DataLoader(preparar(val_ds), batch_size=32)

In [ ]:
# ====================== COMPLETAR ======================
# ========================================================

---
## Ejercicio 3: Transformers en visión (ViT)

Un **ViT** parte la imagen en _patches_ de 16×16, los proyecta como "tokens" y les aplica el mismo Transformer que a un texto. Cargamos un ViT preentrenado en ImageNet.

- a) Clasificar algunas imágenes de CIFAR-10 (reescaladas a 224×224) y mostrar las 3 clases más probables según ImageNet.
- b) Extraer los **pesos de atención** de la última capa y visualizar, para el token `[CLS]`, a qué zonas de la imagen atiende (promediando sobre las cabezas).

In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from datasets import load_dataset

procesador = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
vit = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224", attn_implementation="eager").to(device)  # eager: expone la atención

# CIFAR-10 desde el hub de HF (el servidor de Toronto es lento/inestable)
cifar = load_dataset("uoft-cs/cifar10", split="test[:64]")

# ====================== COMPLETAR ======================
# ========================================================

In [ ]:
# b) mapa de atención del token [CLS] en la última capa
img = cifar[25]["img"]
entrada = procesador(images=img, return_tensors="pt").to(device)
with torch.no_grad():
    salida = vit(**entrada, output_attentions=True)

# attentions: tupla de 12 capas, cada una (batch, cabezas, tokens, tokens)
att = salida.attentions[-1][0]           # última capa -> (12, 197, 197)
att_cls = att[:, 0, 1:].mean(0)          # atención de [CLS] a los 196 patches
mapa = att_cls.reshape(14, 14).cpu().numpy()

fig, axs = plt.subplots(1, 3, figsize=(11, 3.5))
axs[0].imshow(img); axs[0].set_title("imagen"); axs[0].axis("off")
axs[1].imshow(mapa, cmap="viridis"); axs[1].set_title("atención de [CLS] (14×14)")
axs[1].axis("off")
axs[2].imshow(img.resize((224, 224)))
axs[2].imshow(np.kron(mapa, np.ones((16, 16))), cmap="viridis", alpha=0.55)
axs[2].set_title("superpuesta"); axs[2].axis("off")
plt.show()